# Song Test List
Builds a small list of 20 randomly selected songs with their genres and valid tags for testing.

In [21]:
import json
import csv
import random
from collections import defaultdict

DATA_DIR    = './data/'
CSV_DIR     = DATA_DIR + 'csv/'
SONGS_PATH  = CSV_DIR + 'songs.csv'
TAGS_PATH   = CSV_DIR + 'tags.csv'
GENRE_EMB   = DATA_DIR + 'embeddings/genre_embeddings.json'
TAG_EMB     = DATA_DIR + 'embeddings/tag_embeddings.json'
OUTPUT_PATH = DATA_DIR + 'test_songs.json'

N_SONGS  = 20
MIN_TAGS = 3
SEED     = 32

In [22]:
# Load valid vocab from embeddings
with open(GENRE_EMB) as f:
    valid_genres = set(json.load(f).keys())

with open(TAG_EMB) as f:
    valid_tags = set(json.load(f).keys())

print(f"Valid genres: {len(valid_genres)}")
print(f"Valid tags:   {len(valid_tags)}")

Valid genres: 1305
Valid tags:   985


In [23]:
def read_csv(path):
    with open(path, encoding='utf-8', errors='replace') as f:
        reader = csv.reader(f, delimiter=';')
        raw_headers = next(reader)
        headers = [h.strip().strip('"') for h in raw_headers[0].split(',')]
        return [dict(zip(headers, [p.strip().strip('"') for p in row])) for row in reader if len(row) == len(headers)]

song_meta   = {}
song_genres = defaultdict(set)

for row in read_csv(SONGS_PATH):
    sid   = row['spotify_id']
    genre = row['genre_name']
    song_meta[sid] = {'name': row['name'], 'artist': row['artist']}
    if genre in valid_genres:
        song_genres[sid].add(genre)

print(f"Loaded {len(song_meta):,} songs")

Loaded 203,842 songs


In [24]:
song_tags = defaultdict(list)

for row in read_csv(TAGS_PATH):
    sid = row['song_spotify_id']
    tag = row['tag'].lower().strip()
    pop = int(row['popularity']) if row['popularity'].isdigit() else 0
    if tag in valid_tags:
        song_tags[sid].append((tag, pop))

for sid in song_tags:
    seen = {}
    for tag, pop in song_tags[sid]:
        if tag not in seen or pop > seen[tag]:
            seen[tag] = pop
    song_tags[sid] = sorted(seen.items(), key=lambda x: -x[1])

print(f"Songs with valid tags: {sum(1 for sid in song_tags if len(song_tags[sid]) >= MIN_TAGS):,}")

Songs with valid tags: 21,142


In [25]:
candidates = [
    sid for sid in song_meta
    if len(song_genres.get(sid, [])) > 0
    and len(song_tags.get(sid, [])) >= MIN_TAGS
]

random.seed(SEED)
selected = random.sample(candidates, min(N_SONGS, len(candidates)))
print(f"Selected {len(selected)} songs from {len(candidates):,} candidates")

Selected 20 songs from 19,720 candidates


In [26]:
results = []
for sid in selected:
    entry = {
        'spotify_id': sid,
        'name':       song_meta[sid]['name'],
        'artist':     song_meta[sid]['artist'],
        'genres':     sorted(song_genres[sid]),
        'tags':       song_tags[sid][:10],
    }
    results.append(entry)
    print(f"\n{entry['name']} — {entry['artist']}")
    print(f"  genres: {entry['genres']}")
    print(f"  tags:   {entry['tags']}")

with open(OUTPUT_PATH, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nSaved to {OUTPUT_PATH}")


Why Didn't I Think Of That — Doug Stone
  genres: ['country']
  tags:   [('country', 100), ('slow', 8), ('seebest', 4), ('midbest', 4), ('everbest', 4), ('romantic', 4), ('sleepy', 4), ('bestallof', 4), ('dance', 4), ('dbest', 4)]

On the Sunny Side of the Street — Johnny Hodges
  genres: ['bebop', 'big band', 'contemporary post-bop', 'cool jazz', 'hard bop', 'jazz', 'new orleans jazz', 'soul jazz', 'stride', 'swing']
  tags:   [('jazz', 100), ('saxophone', 20), ('bebop', 20)]

Una Calle Nos Separa — Nestor En Bloque
  genres: ['cumbia pop', 'cumbia villera']
  tags:   [('cumbia', 100), ('indie', 34), ('pop', 34), ('alternative', 34)]

You Are The Sunshine Of My Life — Dorothy Donegan
  genres: ['stride']
  tags:   [('soul', 50), ('piano', 50), ('jazz', 50)]

About You Now — Timo Räisänen
  genres: ['swedish alternative rock']
  tags:   [('pop', 100), ('electronic', 40), ('britpop', 40), ('nochmal', 20), ('indie', 20), ('punkd', 20)]

Katja — Apparatschik
  genres: ['balkan brass']
  